# 04 · Validate — geometry preservation, method comparison, pocket & substrate scope

**Standard slot:** *validate (in silico).* **For Project 21 this is the benchmark:** the
**catalytic-geometry preservation rate**, a **scaffolding-method comparison** (RFdiffusion2 vs
Riff-Diff vs motif scaffolding) `[extension]`, and the **pocket-accessibility / substrate-scope** and
active-site-MD figures (D3 pt2).

Needs `results/campaign.csv` (+ `results/ranked.csv` from notebook 03).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Catalytic-geometry preservation — the headline figure
Distribution of catalytic-geometry RMSD vs the 0.5 Å pass bar. The fraction left of the line is the
**preservation rate** — the metric that most distinguishes scaffolding methods. (Numbers here are
SYNTHETIC mock values; on Colab they come from real AF2 predictions.)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

camp = pd.read_csv("results/campaign.csv")
cut = 0.5

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.hist(camp["catalytic_geom_rmsd"], bins=20)
ax.axvline(cut, color="k", ls="--", lw=1, label=f"pass < {cut} A")
ax.set_xlabel("catalytic-geometry RMSD vs theozyme (A)  [SYNTHETIC]")
ax.set_ylabel("designs"); ax.set_title("Catalytic-geometry preservation (triad + oxyanion hole)")
ax.legend(); plt.tight_layout()
plt.savefig("results/catalytic_geometry_hist.png", dpi=150); plt.show()

rate = 100 * (camp["catalytic_geom_rmsd"] <= cut).mean()
print(f"overall catalytic-geometry preservation rate = {rate:.1f}%  [SYNTHETIC demo]")

## 2 · Scaffolding-method comparison `[extension]`
Compare the preservation rate (and hit rate) across scaffolding methods. In the real campaign these
are RFdiffusion2 vs Riff-Diff vs classic RFdiffusion motif scaffolding; here a single mock method is
present, so this cell shows the *shape* of the comparison you will populate on Colab.

In [ ]:
by_method = (camp.assign(pass_geom=camp["catalytic_geom_rmsd"] <= 0.5)
                 .groupby("scaffold_method")
                 .agg(n=("design_id", "size"),
                      geom_pass_rate=("pass_geom", "mean"),
                      mean_plddt_cat=("plddt_catalytic", "mean"))
                 .reset_index())
by_method["geom_pass_rate"] = (100 * by_method["geom_pass_rate"]).round(1)
print("Scaffolding-method comparison (populate with real methods on Colab):")
print(by_method.to_string(index=False))
print("\n[SYNTHETIC] On Colab: compare rfdiffusion2 vs riffdiff vs rfdiffusion on the SAME theozyme.")

## 3 · Pocket accessibility + substrate scope (the project's emphasis)
Geometry is necessary but not sufficient: the ester must physically fit the pocket and orient its
carbonyl toward Ser-OG (docking = **pocket accessibility**), and the **acyl-chain length** the pocket
accepts sets esterase- vs lipase-like **substrate scope**. Here we plot the geometry-passing survivors
by pocket accessibility, and show the acyl-chain scope scan for the best design.

In [ ]:
import matplotlib.pyplot as plt
from enzyme_tools import substrate_scope_scan

# Pocket accessibility among geometry-passing designs (SYNTHETIC flags from the campaign CSV).
geo = camp[camp["catalytic_geom_rmsd"] <= 0.5].copy()
n_acc = int(geo["pose_in_pocket"].sum()) if "pose_in_pocket" in geo else 0
print(f"of {len(geo)} geometry-passing designs, {n_acc} have an ACCESSIBLE pocket "
      f"(ester docks in pocket) [SYNTHETIC]")

# Acyl-chain substrate-scope scan for the best (lowest catalytic-geom) design [extension].
best_id = camp.sort_values("catalytic_geom_rmsd").iloc[0]["design_id"]
scope = substrate_scope_scan(f"results/pred/{best_id}.pdb", acyl_lengths=(2, 4, 6, 8, 10))
lengths = [r["acyl_length"] for r in scope]
fits = [1 if r["fits_and_oriented"] else 0 for r in scope]
scores = [r["vina_score"] for r in scope]

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.bar([str(c) for c in lengths], scores, color=["#4c72b0" if f else "#cccccc" for f in fits])
ax.set_xlabel("acyl-chain length (C-number of pNP-ester)  [SYNTHETIC]")
ax.set_ylabel("Vina fit score (more negative = better)")
ax.set_title(f"Substrate scope of {best_id}\n(blue = fits & oriented toward Ser-OG)")
plt.tight_layout(); plt.savefig("results/substrate_scope.png", dpi=150); plt.show()
pref = [c for c, f in zip(lengths, fits) if f]
print(f"predicted-acceptable acyl chains for {best_id}: C{pref}  [SYNTHETIC] "
      "(short = esterase-like, longer = lipase-like)")

## 4 · Active-site MD stability (top candidates)
Short MD (OpenMM) checks the triad + pocket don't drift apart. Plot MD RMSD vs catalytic geometry for
the ranked survivors as orthogonal evidence — a triad that drifts will not catalyse even with perfect
static geometry.

In [ ]:
import matplotlib.pyplot as plt
# ranked.csv comes from the shared filter (fp.Design fields); vina_score lives in campaign.csv,
# so merge it back by design_id for the stability view.
try:
    ranked = pd.read_csv("results/ranked.csv")
    ranked = ranked.merge(camp[["design_id", "vina_score"]], on="design_id", how="left")
except FileNotFoundError:
    ranked = camp.copy()

top = ranked.head(min(20, len(ranked)))
fig, ax = plt.subplots(figsize=(5.2, 3.4))
sc = ax.scatter(top["vina_score"], top["md_rmsd"],
                c=top["catalytic_geom_rmsd"], cmap="viridis")
ax.set_xlabel("Vina ester-fit score (more negative = better fit)  [SYNTHETIC]")
ax.set_ylabel("active-site MD RMSD (A)  [SYNTHETIC]")
ax.set_title("Top candidates: pocket fit vs active-site stability")
fig.colorbar(sc, label="catalytic-geom RMSD (A)")
plt.tight_layout(); plt.savefig("results/docking_md.png", dpi=150); plt.show()
print("Lower-left + dark points (good fit, stable, good geometry) are the best candidates [SYNTHETIC].")

## 5 · Honest hit-rate accounting
Report N(pass all layers) / N(generated), and remind the reader of the field reality: even a good
preservation rate is **not** an activity rate. Geometry ≠ catalysis; a pNP-ester kinetic assay (with
the Ser→Ala dead mutant) is required.

In [ ]:
n_total = len(camp)
try:
    ranked = pd.read_csv("results/ranked.csv")
    n_hits = int((ranked["layers_passed"] >= 3).sum())
except Exception:
    n_hits = int((camp["catalytic_geom_rmsd"] <= 0.5).sum())
print(f"Hit-rate accounting [SYNTHETIC demo]:")
print(f"  generated            : {n_total}")
print(f"  pass all filter layers: {n_hits}  ({100*n_hits/max(n_total,1):.1f}%)")
print("\nREALITY CHECK: de novo enzyme activity rates are <5% without directed evolution, and")
print("in-silico catalytic geometry does NOT guarantee activity. Only a kinetic assay decides.")

## D3 (part 2) checklist
- [ ] Catalytic-geometry preservation histogram (`results/catalytic_geometry_hist.png`) + rate.
- [ ] Scaffolding-method comparison table/figure (real methods on Colab) `[extension]`.
- [ ] Pocket-accessibility count + substrate-scope figure (`results/substrate_scope.png`) `[extension]`.
- [ ] Active-site-MD figure on the ranked top set.
- [ ] Honest hit-rate accounting with the "geometry ≠ activity" caveat stated.

**Next:** `05_validation_plan.ipynb` — the pNP-ester kinetic-assay plan + controls + enantioselectivity stretch.